In [2]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
tqdm.pandas()

In [3]:
DEVICE = "mps" if torch.mps.is_available() else "cpu" 

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

mps


In [4]:
res_net = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1).to(DEVICE)

In [5]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

In [6]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((256, 256))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        
        return img_tensor

In [7]:
tr_images_ds = ImageDataset(IMAGE_FOLDER, train_df)

batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [9]:
sureness = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure = res_net(batch.to(DEVICE))

    sure = image_sure.to('cpu').numpy()

    sureness.extend(sure)

100%|██████████| 1065/1065 [04:49<00:00,  3.67it/s]


In [8]:
DATA_PATH = "/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data"

train_text_emb = np.load(f'{DATA_PATH}/train_text_embeddings.npy', allow_pickle=True).tolist()
train_img_emb = np.load(f'{DATA_PATH}/train_img_embedings.npy', allow_pickle=True).tolist()

test_text_emb = np.load(f'{DATA_PATH}/test_text_embeddings.npy', allow_pickle=True).tolist()
test_img_emb = np.load(f'{DATA_PATH}/test_img_embedings.npy', allow_pickle=True).tolist()

In [10]:
for i in range(len(train_text_emb[-1])):
    train_df[f"text{i}"] = [train_text_emb[j][i] for j in range(len(train_text_emb))]

for i in range(len(train_img_emb[-1])):
    train_df[f"img{i}"] = [train_img_emb[j][i] for j in range(len(train_img_emb))]

for i in range(1000):
    train_df[f"f{i}"] = [sureness[j][i] for j in range(len(sureness))]

In [37]:
features = [f'f{i}' for i in range(1000)]
# features += [f'img{i}' for i in range(len(train_img_emb[0]))]
features += [f'text{i}' for i in range(len(train_text_emb[0]))]
# features.append('description')

target = 'label'

In [38]:
X_tr, X_val, y_tr, y_val = train_test_split(train_df[features], train_df[target], test_size=0.3, shuffle=True, stratify=train_df[target])

In [30]:
catboost_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.01,
    depth=6,                  # Deeper, but heavily regularized
    l2_leaf_reg=20,
    rsm=0.2,                  # Only look at 600 features per split!
    random_strength=5.0,      # Heavy penalty for "too perfect" splits
    bagging_temperature=2.0,  # Aggressive row subsampling
    min_data_in_leaf=50,      # Prevents tiny leaves fitting to noise
    early_stopping_rounds=100,
    eval_metric='AUC',
    verbose=200,
    text_features=['description'],
    random_state=67,
    leaf_estimation_method='Newton'   
)


catboost_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

pred = catboost_model.predict(X_val)
print("Точность на валидационной выборке:", accuracy_score(pred, y_val))

pred = catboost_model.predict_proba(X_val).tolist()
print("ROC AUC на валидационной выборке:", roc_auc_score(y_val, [i[1] for i in pred]))

0:	test: 0.6844802	best: 0.6844802 (0)	total: 50.2ms	remaining: 4m 11s
200:	test: 0.8904514	best: 0.8904514 (200)	total: 9.49s	remaining: 3m 46s
400:	test: 0.9008542	best: 0.9008542 (400)	total: 19s	remaining: 3m 38s
600:	test: 0.9081951	best: 0.9081951 (600)	total: 29s	remaining: 3m 31s
800:	test: 0.9129499	best: 0.9129499 (800)	total: 39s	remaining: 3m 24s
1000:	test: 0.9172105	best: 0.9172107 (998)	total: 49.2s	remaining: 3m 16s
1200:	test: 0.9226045	best: 0.9226045 (1200)	total: 59.5s	remaining: 3m 8s
1400:	test: 0.9272594	best: 0.9272594 (1400)	total: 1m 9s	remaining: 2m 59s
1600:	test: 0.9300647	best: 0.9300647 (1600)	total: 1m 19s	remaining: 2m 49s
1800:	test: 0.9322174	best: 0.9322174 (1800)	total: 1m 30s	remaining: 2m 40s
2000:	test: 0.9337719	best: 0.9337750 (1997)	total: 1m 41s	remaining: 2m 31s
2200:	test: 0.9351844	best: 0.9351844 (2200)	total: 1m 51s	remaining: 2m 22s
2400:	test: 0.9362815	best: 0.9362815 (2400)	total: 2m 2s	remaining: 2m 12s
2600:	test: 0.9372033	best: 0

In [86]:
catboost_model.save_model('catboost_model.cbm')

In [ ]:
ts_images_ds = ImageDataset(IMAGE_FOLDER, test_df)

In [ ]:
images_ts = DataLoader(ts_images_ds, batch_size=batch_size, shuffle=False)

In [ ]:
sureness_ts = []

for images_ in tqdm(images_ts):
    with torch.no_grad():
        sure = res_net(images_.to(DEVICE))

    sureness_ts.extend(sure.to('cpu').numpy().tolist())

100%|██████████| 264/264 [01:20<00:00,  3.28it/s]


In [21]:
for i in range(1000):
    test_df[f"f{i}"] = [sureness_ts[j][i] for j in range(len(sureness_ts))]

for i in range(len(train_text_emb[-1])):
    test_df[f"text{i}"] = [test_text_emb[j][i] for j in range(len(test_text_emb))]

for i in range(len(train_img_emb[-1])):
    test_df[f"img{i}"] = [test_img_emb[j][i] for j in range(len(test_img_emb))]

In [87]:
pred_ts = catboost_model.predict_proba(test_df[features])

In [88]:
test_df['y_pred'] = [i[1] for i in pred_ts.tolist()]

In [89]:
(test_df[["id", 'y_pred']]).to_csv("submission_with_resnet.csv", index=False)